In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("cleaned_students_dropout01.csv")

print("=== Data Overview ===")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print("\nColumns and Data Types:")
print(df.dtypes)

print("\nNumerical Summary Statistics:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())


=== Data Overview ===
Shape: 10198 rows × 10 columns

Columns and Data Types:
School_Type             object
Location                object
Infrastructure          object
Teaching_Staff          object
Gender                  object
Caste                   object
Age                      int64
Standard                 int64
Socioeconomic_Status    object
Dropout_Status          object
dtype: object

Numerical Summary Statistics:
                Age      Standard
count  10198.000000  10198.000000
mean      12.778388      8.735634
std        1.758911      1.682079
min       10.000000      5.000000
25%       11.000000      8.000000
50%       13.000000      9.000000
75%       14.000000     10.000000
max       16.000000     12.000000

Missing Values:
School_Type             0
Location                0
Infrastructure          0
Teaching_Staff          0
Gender                  0
Caste                   0
Age                     0
Standard                0
Socioeconomic_Status    0
Dropout_St

In [7]:
print("\nGenerating Univariate Plots...")

# Define a nice color palette (Plotly qualitative Set2)
colors = px.colors.qualitative.Set2

# Numerical distributions (Histogram + KDE + Box)
fig_num = make_subplots(rows=2, cols=2,
                        subplot_titles=("Distribution of Age", "Boxplot of Age",
                                        "Distribution of Standard", "Boxplot of Standard"))

fig_num.add_trace(go.Histogram(x=df['Age'], name='Age', marker_color=colors[0], nbinsx=7), row=1, col=1)
fig_num.add_trace(go.Box(y=df['Age'], name='Age', marker_color=colors[0]), row=1, col=2)

fig_num.add_trace(go.Histogram(x=df['Standard'], name='Standard', marker_color=colors[1], nbinsx=8), row=2, col=1)
fig_num.add_trace(go.Box(y=df['Standard'], name='Standard', marker_color=colors[1]), row=2, col=2)

fig_num.update_layout(height=900, title_text="Interactive Univariate Numerical Distributions")
fig_num.write_html("univariate_numerical.html")
fig_num.show()


Generating Interactive Univariate Plots...


In [14]:
# Categorical frequency bars (fixed with proper colors)
cat_cols = ['School_Type', 'Location', 'Gender', 'Caste', 'Socioeconomic_Status', 'Dropout_Status']
fig_cat = make_subplots(rows=3, cols=2, subplot_titles=[f'Frequency: {col}' for col in cat_cols])

for i, col in enumerate(cat_cols):
    counts = df[col].value_counts().reset_index()
    counts.columns = [col, 'count']
    row = i // 2 + 1
    col_pos = i % 2 + 1
    fig_cat.add_trace(
        go.Bar(x=counts[col], y=counts['count'], name=col,
               marker_color=colors[i % len(colors)]),
        row=row, col=col_pos
    )

fig_cat.update_layout(height=1200, title_text="Categorical Frequency Counts", barmode='group')
fig_cat.write_html("univariate_categorical.html")
fig_cat.show()

In [13]:
# Extra categorical: Infrastructure & Teaching_Staff
fig_extra = make_subplots(rows=1, cols=2, subplot_titles=("Infrastructure", "Teaching_Staff"))
extra_cols = ['Infrastructure', 'Teaching_Staff']
for i, col in enumerate(extra_cols):
    counts = df[col].value_counts().reset_index()
    counts.columns = [col, 'count']
    fig_extra.add_trace(
        go.Bar(x=counts[col], y=counts['count'],
               marker_color=colors[i % len(colors)]),
        row=1, col=i+1
    )

fig_extra.update_layout(height=600, title_text="Extra Categorical Counts")
fig_extra.write_html("univariate_categorical_extra.html")
fig_extra.show()

In [12]:
print("Generating Bivariate & Multivariate Plots...")

# Boxplots: Numerical vs Target
fig_box = make_subplots(rows=1, cols=2, subplot_titles=("Age by Dropout Status", "Standard by Dropout Status"))
fig_box.add_trace(go.Box(x=df['Dropout_Status'], y=df['Age'], name='Age', marker_color=colors[0]), row=1, col=1)
fig_box.add_trace(go.Box(x=df['Dropout_Status'], y=df['Standard'], name='Standard', marker_color=colors[1]), row=1, col=2)
fig_box.update_layout(height=600, title_text="Boxplots: Numerical vs Target")
fig_box.write_html("bivariate_boxplots.html")
fig_box.show()

Generating Bivariate & Multivariate Plots...


In [15]:
# Violin plots
fig_violin = make_subplots(rows=1, cols=2, subplot_titles=("Age Distribution by Dropout Status", "Standard Distribution by Dropout Status"))
fig_violin.add_trace(go.Violin(x=df['Dropout_Status'], y=df['Age'], name='Age', marker_color=colors[0], box_visible=True), row=1, col=1)
fig_violin.add_trace(go.Violin(x=df['Dropout_Status'], y=df['Standard'], name='Standard', marker_color=colors[1], box_visible=True), row=1, col=2)
fig_violin.update_layout(height=600, title_text="Interactive Violin Plots")
fig_violin.write_html("bivariate_violins.html")
fig_violin.show()

In [17]:
# Categorical vs Target (grouped bars - fully interactive)
cat_vs_target = ['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender', 'Caste']
fig_cat_target = make_subplots(rows=3, cols=2, subplot_titles=[f'{col} vs Dropout' for col in cat_vs_target])

for i, col in enumerate(cat_vs_target):
    row = i // 2 + 1
    col_pos = i % 2 + 1
    grouped = df.groupby([col, 'Dropout_Status']).size().reset_index(name='count')
    for status in grouped['Dropout_Status'].unique():
        subset = grouped[grouped['Dropout_Status'] == status]
        fig_cat_target.add_trace(
            go.Bar(x=subset[col], y=subset['count'], name=status,
                   marker_color=colors[0] if status == 'Dropout' else colors[1]),
            row=row, col=col_pos
        )

fig_cat_target.update_layout(height=1400, title_text="Categorical Features vs Dropout Status", barmode='group')
fig_cat_target.write_html("bivariate_categorical_vs_target.html")
fig_cat_target.show()

In [19]:
# SES vs Dropout (special focus)
fig_ses = px.bar(df.groupby(['Socioeconomic_Status', 'Dropout_Status']).size().reset_index(name='count'),
                 x='Socioeconomic_Status', y='count', color='Dropout_Status',
                 color_discrete_map={'Dropout': colors[0], 'Enrolled': colors[1]},
                 title="Socioeconomic Status vs Dropout Status")
fig_ses.write_html("bivariate_ses_vs_dropout.html")
fig_ses.show()

In [22]:
# Correlation heatmap
df_encoded = df.copy()
df_encoded['Dropout_Status_Encoded'] = df_encoded['Dropout_Status'].map({'Dropout': 1, 'Enrolled': 0})
corr_matrix = df_encoded[['Age', 'Standard', 'Dropout_Status_Encoded']].corr().round(2)

fig_heatmap = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdYlBu_r',
    text=corr_matrix.values,
    texttemplate="%{text}",
    hovertemplate="%{y} vs %{x}<br>Correlation: %{z}<extra></extra>"
))
fig_heatmap.update_layout(title="Correlation Heatmap (Numerical + Encoded Target)", height=600)
fig_heatmap.write_html("correlation_heatmap.html")
fig_heatmap.show()

In [23]:
# ====================== 4. FEATURE INSIGHTS ======================
print("\n=== Key Dropout Rates by Category ===")
for col in ['School_Type', 'Socioeconomic_Status', 'Gender', 'Caste', 'Infrastructure', 'Location']:
    print(f"\n{col}:")
    print(df.groupby(col)['Dropout_Status'].value_counts(normalize=True).unstack() * 100)


=== Key Dropout Rates by Category ===

School_Type:
Dropout_Status    Dropout   Enrolled
School_Type                         
Government      86.340487  13.659513
Private         26.574468  73.425532

Socioeconomic_Status:
Dropout_Status          Dropout   Enrolled
Socioeconomic_Status                      
High                   4.011260  95.988740
Low                   95.838696   4.161304
Moderate              52.487008  47.512992

Gender:
Dropout_Status    Dropout   Enrolled
Gender                              
Female          75.487744  24.512256
Male            48.032258  51.967742

Caste:
Dropout_Status    Dropout   Enrolled
Caste                               
General         67.036011  32.963989
OBC             32.264398  67.735602
SC              68.035191  31.964809
ST              74.256055  25.743945

Infrastructure:
Dropout_Status    Dropout   Enrolled
Infrastructure                      
Basic           68.406530  31.593470
Excellent       28.054662  71.945338
Good     